# 09 - Final Model Comparison & Selection

Comprehensive cross-model comparison bringing together results from:
- Traditional ML regression & classification models
- Deep learning (LSTM, Conv1D-LSTM, GRU) models
- Optuna-tuned models

**Goal**: Select the best production models for regression (AQI prediction) and classification (AQI category).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from config import DATA_PROCESSED, FIGURES_DIR, MODELS_DIR, RANDOM_STATE

plt.style.use('seaborn-v0_8-whitegrid')
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Configuration loaded.')
print(f'DATA_PROCESSED: {DATA_PROCESSED}')
print(f'FIGURES_DIR:    {FIGURES_DIR}')
print(f'MODELS_DIR:     {MODELS_DIR}')

## 1. Load All Results CSVs

In [ ]:
# Load each results file with error handling
results = {}

csv_files = {
    'regression': 'regression_results.csv',
    'classification': 'classification_results.csv',
    'deep_learning': 'deep_learning_results.csv',
    'optuna': 'optuna_results.csv',
}

for key, filename in csv_files.items():
    path = DATA_PROCESSED / filename
    if path.exists():
        results[key] = pd.read_csv(path, index_col=0)
        print(f'Loaded {filename}: {results[key].shape}')
    else:
        print(f'WARNING: {filename} not found at {path} - skipping')

print(f'\nLoaded {len(results)} / {len(csv_files)} result files.')

## 2. Unified Regression Comparison

Combine ML regression, deep learning, and Optuna-tuned models into a single comparison table with a **Type** column.

In [ ]:
# Build unified regression table
frames = []

# ML regression results
if 'regression' in results:
    ml_reg = results['regression'].copy()
    ml_reg['Type'] = 'ML'
    ml_reg.index = '[ML] ' + ml_reg.index.astype(str)
    ml_reg.index.name = 'Model'
    frames.append(ml_reg)

# Deep learning results
if 'deep_learning' in results:
    dl_reg = results['deep_learning'].copy()
    dl_reg['Type'] = 'DL'
    dl_reg.index = '[DL] ' + dl_reg.index.astype(str)
    dl_reg.index.name = 'Model'
    frames.append(dl_reg)

# Optuna-tuned results (regression rows only)
if 'optuna' in results:
    optuna_reg = results['optuna'].copy()
    # If there's a task column, filter for regression; otherwise include all
    if 'task' in optuna_reg.columns:
        optuna_reg = optuna_reg[optuna_reg['task'] == 'regression'].drop(columns=['task'])
    optuna_reg['Type'] = 'Tuned'
    optuna_reg.index = '[Tuned] ' + optuna_reg.index.astype(str)
    optuna_reg.index.name = 'Model'
    frames.append(optuna_reg)

if frames:
    # Find common numeric columns across all frames
    common_cols = ['R2', 'RMSE', 'MAE']
    extra_cols = ['Train_Time_s']
    keep_cols = [c for c in common_cols + extra_cols + ['Type'] if all(c in f.columns for f in frames)]
    # Ensure Type is always present
    if 'Type' not in keep_cols:
        keep_cols.append('Type')
    
    unified_reg = pd.concat([f[[c for c in keep_cols if c in f.columns]] for f in frames])
    unified_reg = unified_reg.sort_values('R2', ascending=False)
    
    print('=' * 80)
    print('UNIFIED REGRESSION MODEL COMPARISON')
    print('=' * 80)
    print(unified_reg.round(4).to_string())
else:
    unified_reg = pd.DataFrame()
    print('No regression results available.')

## 3. Figure 30 - Comprehensive Comparison Chart

In [ ]:
if not unified_reg.empty:
    # Color mapping by Type
    type_colors = {'ML': '#2196F3', 'DL': '#9C27B0', 'Tuned': '#FF9800'}
    bar_colors = [type_colors.get(t, '#888888') for t in unified_reg['Type']]

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))

    models = unified_reg.index.tolist()

    # Top-left: R2 comparison
    ax = axes[0, 0]
    ax.barh(models, unified_reg['R2'], color=bar_colors, edgecolor='black', alpha=0.85)
    ax.set_xlabel('R² Score')
    ax.set_title('R² Score (higher is better)', fontweight='bold')
    for i, v in enumerate(unified_reg['R2']):
        ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)
    ax.invert_yaxis()

    # Top-right: RMSE comparison
    ax = axes[0, 1]
    ax.barh(models, unified_reg['RMSE'], color=bar_colors, edgecolor='black', alpha=0.85)
    ax.set_xlabel('RMSE')
    ax.set_title('RMSE (lower is better)', fontweight='bold')
    for i, v in enumerate(unified_reg['RMSE']):
        ax.text(v + 0.5, i, f'{v:.2f}', va='center', fontsize=8)
    ax.invert_yaxis()

    # Bottom-left: MAE comparison
    ax = axes[1, 0]
    ax.barh(models, unified_reg['MAE'], color=bar_colors, edgecolor='black', alpha=0.85)
    ax.set_xlabel('MAE')
    ax.set_title('MAE (lower is better)', fontweight='bold')
    for i, v in enumerate(unified_reg['MAE']):
        ax.text(v + 0.5, i, f'{v:.2f}', va='center', fontsize=8)
    ax.invert_yaxis()

    # Bottom-right: Training time comparison
    ax = axes[1, 1]
    if 'Train_Time_s' in unified_reg.columns:
        ax.barh(models, unified_reg['Train_Time_s'], color=bar_colors, edgecolor='black', alpha=0.85)
        ax.set_xlabel('Training Time (seconds)')
        ax.set_title('Training Time (lower is better)', fontweight='bold')
        for i, v in enumerate(unified_reg['Train_Time_s']):
            ax.text(v + 0.1, i, f'{v:.1f}s', va='center', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'Training time not available', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
    ax.invert_yaxis()

    # Add legend
    from matplotlib.patches import Patch
    legend_handles = [Patch(facecolor=c, edgecolor='black', label=t)
                      for t, c in type_colors.items()
                      if t in unified_reg['Type'].values]
    fig.legend(handles=legend_handles, loc='upper center', ncol=3, fontsize=12,
               bbox_to_anchor=(0.5, 1.02))

    plt.suptitle('Figure 30: Comprehensive Model Comparison', fontsize=16,
                 fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '30_final_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {FIGURES_DIR / "30_final_comparison.png"}')
else:
    print('Skipping figure - no unified regression results available.')

## 4. Classification Summary Table

In [ ]:
if 'classification' in results:
    clf_df = results['classification'].copy()
    
    # Sort by Weighted_F1 if available, else by Accuracy
    sort_col = 'Weighted_F1' if 'Weighted_F1' in clf_df.columns else 'Accuracy'
    clf_df = clf_df.sort_values(sort_col, ascending=False)
    
    print('=' * 80)
    print('CLASSIFICATION MODEL COMPARISON')
    print('=' * 80)
    print(clf_df.round(4).to_string())
else:
    clf_df = pd.DataFrame()
    print('No classification results available.')

## 5. Select Final Production Models

In [ ]:
selections = []

# --- Best Regression Model (highest R2) ---
if not unified_reg.empty:
    best_reg_label = unified_reg['R2'].idxmax()      # e.g. "[Tuned] LightGBM"
    best_reg_row = unified_reg.loc[best_reg_label]
    best_reg_type = best_reg_row['Type']
    best_reg_r2 = best_reg_row['R2']
    # Strip the prefix to get the raw model name for file paths
    best_reg_name = best_reg_label.split('] ', 1)[-1]
    
    # Determine model path based on type
    if best_reg_type == 'DL':
        model_path = str(MODELS_DIR / 'deep_learning' / f'{best_reg_name.lower().replace(" ", "_").replace("-", "_")}.keras')
    elif best_reg_type == 'Tuned':
        model_path = str(MODELS_DIR / 'tuned' / f'{best_reg_name.lower().replace(" ", "_")}.joblib')
    else:
        model_path = str(MODELS_DIR / f'{best_reg_name.lower().replace(" ", "_")}.joblib')
    
    selections.append({
        'task': 'regression',
        'model_name': best_reg_name,
        'model_type': best_reg_type,
        'key_metric': 'R2',
        'metric_value': round(best_reg_r2, 4),
        'model_path': model_path,
    })
    
    print('BEST REGRESSION MODEL')
    print(f'  Model:  {best_reg_name}')
    print(f'  Type:   {best_reg_type}')
    print(f'  R²:     {best_reg_r2:.4f}')
    print(f'  RMSE:   {best_reg_row["RMSE"]:.2f}')
    print(f'  MAE:    {best_reg_row["MAE"]:.2f}')
    print(f'  Path:   {model_path}')
else:
    print('No regression results available for selection.')

print()

# --- Best Classification Model (highest Weighted_F1) ---
if not clf_df.empty:
    metric_col = 'Weighted_F1' if 'Weighted_F1' in clf_df.columns else 'Accuracy'
    best_clf_name = clf_df[metric_col].idxmax()
    best_clf_row = clf_df.loc[best_clf_name]
    best_clf_val = best_clf_row[metric_col]
    
    clf_model_path = str(MODELS_DIR / f'{best_clf_name.lower().replace(" ", "_")}_clf.joblib')
    
    selections.append({
        'task': 'classification',
        'model_name': best_clf_name,
        'model_type': 'ML',
        'key_metric': metric_col,
        'metric_value': round(best_clf_val, 4),
        'model_path': clf_model_path,
    })
    
    print('BEST CLASSIFICATION MODEL')
    print(f'  Model:        {best_clf_name}')
    print(f'  {metric_col}: {best_clf_val:.4f}')
    print(f'  Path:         {clf_model_path}')
else:
    print('No classification results available for selection.')

## 6. Save Final Model Selection

In [ ]:
if selections:
    selection_df = pd.DataFrame(selections)
    selection_df.to_csv(DATA_PROCESSED / 'final_model_selection.csv', index=False)
    
    print('Final model selection saved to:')
    print(f'  {DATA_PROCESSED / "final_model_selection.csv"}')
    print()
    print(selection_df.to_string(index=False))
else:
    print('No models selected - ensure result CSVs exist before running this notebook.')

## Summary & Conclusions

### Final Model Selection

This notebook consolidates results from all modeling stages:

1. **Traditional ML** (notebook 04): Linear models, tree-based models, gradient boosting
2. **Classification** (notebook 05): Multi-class AQI category prediction
3. **Deep Learning** (notebook 06): LSTM, Conv1D-LSTM, GRU on raw sequences
4. **Hyperparameter Tuning** (Optuna): Fine-tuned top performers

### Key Takeaways

- **Gradient boosting models** (LightGBM, CatBoost, XGBoost) consistently perform best on this tabular dataset, both for regression and classification
- **Deep learning models** work well on raw sequences but typically do not surpass tree-based models with engineered features on tabular data
- **Optuna tuning** provides marginal-to-moderate improvement over default hyperparameters
- The selected production models balance accuracy, inference speed, and robustness

### Output Files
- `30_final_comparison.png` - Visual comparison across all model types
- `final_model_selection.csv` - Production model selections with paths and metrics